# Notebook 01 — System Baseline Characterization

Capture and visualize the complete system state before any Unheaded services start.
This is the control measurement. Every optimization is measured against this baseline.

**Run before starting any services. Commit results. Tag as baseline-YYYY-MM-DD.**

In [ ]:
import subprocess, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'figure.facecolor':'#0a0a0a','axes.facecolor':'#111111','axes.edgecolor':'#333333',
    'axes.labelcolor':'#c9c9c9','text.color':'#c9c9c9','xtick.color':'#888888',
    'ytick.color':'#888888','grid.color':'#1e1e1e','font.family':'monospace',
    'font.size':10,'axes.titlecolor':'#ffffff',
})
ACCENT,ACCENT2,ACCENT3,WARN='#00ff88','#00aaff','#ffaa00','#ff4444'

def sh(cmd):
    try: return subprocess.check_output(cmd,shell=True,stderr=subprocess.DEVNULL,text=True).strip()
    except: return ""

LIVE = os.environ.get('UNHEADED_LIVE','0')=='1' or os.path.exists('/proc/cpuinfo')
print(f"Mode: {'LIVE' if LIVE else 'SYNTHETIC'}")
np.random.seed(2026)

## 1. CPU Topology

In [ ]:
if LIVE:
    nproc = int(sh("nproc --all") or "8")
    model = sh("grep 'model name' /proc/cpuinfo | head -1 | cut -d: -f2").strip()
    cpu_mhz = [float(x.split(':')[1]) for x in sh("grep 'cpu MHz' /proc/cpuinfo").splitlines() if ':' in x]
else:
    nproc = 16
    model = "AMD Ryzen 9 7950X (synthetic)"
    cpu_mhz = np.random.normal(3500, 400, nproc).clip(2200, 5000).tolist()

print(f"CPUs: {nproc}")
print(f"Model: {model}")
print(f"MHz range: {min(cpu_mhz):.0f} - {max(cpu_mhz):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5)); fig.patch.set_facecolor('#0a0a0a')
ax = axes[0]
bars = ax.bar(range(nproc), cpu_mhz, color=[ACCENT if m > 3000 else ACCENT2 for m in cpu_mhz], alpha=0.85)
ax.axhline(np.mean(cpu_mhz), color=WARN, ls='--', lw=1.5, label=f'Mean {np.mean(cpu_mhz):.0f} MHz')
ax.set_xlabel('CPU Core'); ax.set_ylabel('MHz'); ax.set_title('Per-Core Frequency'); ax.legend(); ax.grid(alpha=0.3,axis='y')

ax = axes[1]
ax.hist(cpu_mhz, bins=20, color=ACCENT, alpha=0.8, edgecolor='#0a0a0a')
ax.set_xlabel('MHz'); ax.set_ylabel('Count'); ax.set_title('Frequency Distribution'); ax.grid(alpha=0.3)
plt.suptitle('CPU Topology', color='white', fontsize=13)
plt.tight_layout(); plt.savefig('/tmp/01_cpu.png', dpi=130, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()

## 2. Memory Layout

In [ ]:
if LIVE:
    with open('/proc/meminfo') as f:
        meminfo = {}
        for line in f:
            k, v = line.split(':')
            meminfo[k.strip()] = int(v.strip().split()[0])
    total_gb = meminfo['MemTotal'] / 1024**2
    avail_gb = meminfo['MemAvailable'] / 1024**2
    cached_gb = meminfo.get('Cached', 0) / 1024**2
    buffers_gb = meminfo.get('Buffers', 0) / 1024**2
    used_gb = total_gb - avail_gb
else:
    total_gb = 64.0
    avail_gb = 58.2
    cached_gb = 3.1
    buffers_gb = 0.8
    used_gb = total_gb - avail_gb

print(f"Total: {total_gb:.1f} GB | Available: {avail_gb:.1f} GB | Used: {used_gb:.1f} GB ({used_gb/total_gb*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 4)); fig.patch.set_facecolor('#0a0a0a')
categories = ['Used\n(OS+procs)', 'Cached\n(reclaimable)', 'Buffers', 'Available']
values = [used_gb - cached_gb - buffers_gb, cached_gb, buffers_gb, avail_gb]
colors = [WARN, ACCENT3, ACCENT2, ACCENT]
left = 0
for cat, val, col in zip(categories, values, colors):
    ax.barh(0, val, left=left, color=col, alpha=0.85, label=f'{cat}: {val:.1f}GB', height=0.5)
    left += val
ax.set_xlim(0, total_gb); ax.set_yticks([]); ax.set_xlabel('GB')
ax.set_title(f'Memory Layout — {total_gb:.0f}GB Total'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.savefig('/tmp/01_mem.png', dpi=130, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()

## 3. Disk I/O Baseline

In [ ]:
if LIVE:
    iostat_out = sh("iostat -x 1 5 2>/dev/null | tail -20")
    lines = [l for l in iostat_out.splitlines() if l and not l.startswith('Linux') and not l.startswith('Device')]
    devices = {}
    for line in lines:
        parts = line.split()
        if parts and not parts[0].startswith('avg'):
            try: devices[parts[0]] = {'r_iops': float(parts[3]), 'w_iops': float(parts[4]), 'util': float(parts[-1])}
            except: pass
else:
    devices = {
        'nvme0n1': {'r_iops': 1250.0, 'w_iops': 340.0, 'util': 0.8},
        'sda':     {'r_iops': 85.0,   'w_iops': 12.0,  'util': 0.3},
    }

if not devices:
    devices = {'nvme0n1': {'r_iops': 0, 'w_iops': 0, 'util': 0}}

print("Disk I/O Baseline:")
for dev, stats in devices.items():
    print(f"  {dev}: r={stats['r_iops']:.0f} IOPS | w={stats['w_iops']:.0f} IOPS | util={stats['util']:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5)); fig.patch.set_facecolor('#0a0a0a')
devnames = list(devices.keys())
r_iops = [devices[d]['r_iops'] for d in devnames]
w_iops = [devices[d]['w_iops'] for d in devnames]
x = np.arange(len(devnames))
axes[0].bar(x-0.2, r_iops, 0.4, color=ACCENT, alpha=0.85, label='Read IOPS')
axes[0].bar(x+0.2, w_iops, 0.4, color=ACCENT2, alpha=0.85, label='Write IOPS')
axes[0].set_xticks(x); axes[0].set_xticklabels(devnames); axes[0].set_ylabel('IOPS')
axes[0].set_title('IOPS by Device (Idle Baseline)'); axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

utils = [devices[d]['util'] for d in devnames]
colors = [WARN if u > 50 else ACCENT for u in utils]
axes[1].bar(devnames, utils, color=colors, alpha=0.85)
axes[1].axhline(80, color=WARN, ls='--', lw=2, label='80% saturation threshold')
axes[1].set_ylabel('Utilization %'); axes[1].set_title('I/O Utilization (Idle)'); axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')
plt.suptitle('Disk I/O Baseline', color='white', fontsize=13)
plt.tight_layout(); plt.savefig('/tmp/01_disk.png', dpi=130, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()

## 4. Network Interfaces

In [ ]:
if LIVE:
    net_lines = sh("cat /proc/net/dev | tail -n +3").splitlines()
    ifaces = {}
    for line in net_lines:
        parts = line.split()
        if parts:
            name = parts[0].rstrip(':')
            if name not in ('lo',):
                try: ifaces[name] = {'rx_bytes': int(parts[1]), 'tx_bytes': int(parts[9]), 'rx_drop': int(parts[4]), 'tx_drop': int(parts[12])}
                except: pass
else:
    ifaces = {
        'eth0':  {'rx_bytes': 4294967296, 'tx_bytes': 2147483648, 'rx_drop': 0, 'tx_drop': 0},
        'wg0':   {'rx_bytes': 134217728,  'tx_bytes': 67108864,   'rx_drop': 0, 'tx_drop': 0},
        'vxlan0':{'rx_bytes': 268435456,  'tx_bytes': 134217728,  'rx_drop': 0, 'tx_drop': 0},
    }

print("Network Interfaces:")
for iface, stats in ifaces.items():
    rx_gb = stats['rx_bytes'] / 1024**3
    tx_gb = stats['tx_bytes'] / 1024**3
    print(f"  {iface}: RX={rx_gb:.2f}GB TX={tx_gb:.2f}GB drops={stats['rx_drop']+stats['tx_drop']}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5)); fig.patch.set_facecolor('#0a0a0a')
names = list(ifaces.keys())
rx = [ifaces[i]['rx_bytes']/1024**3 for i in names]
tx = [ifaces[i]['tx_bytes']/1024**3 for i in names]
x = np.arange(len(names))
axes[0].bar(x-0.2, rx, 0.4, color=ACCENT, alpha=0.85, label='RX GB')
axes[0].bar(x+0.2, tx, 0.4, color=ACCENT2, alpha=0.85, label='TX GB')
axes[0].set_xticks(x); axes[0].set_xticklabels(names); axes[0].set_ylabel('GB transferred')
axes[0].set_title('Cumulative Transfer by Interface'); axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

drops = [ifaces[i]['rx_drop']+ifaces[i]['tx_drop'] for i in names]
colors = [WARN if d > 0 else ACCENT for d in drops]
axes[1].bar(names, drops, color=colors, alpha=0.85)
axes[1].set_ylabel('Total Drops'); axes[1].set_title('Packet Drops (should be 0)'); axes[1].grid(alpha=0.3, axis='y')
plt.suptitle('Network Interface Baseline', color='white', fontsize=13)
plt.tight_layout(); plt.savefig('/tmp/01_net.png', dpi=130, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()

## 5. Process Inventory

In [ ]:
if LIVE:
    proc_count = int(sh("ps aux | wc -l") or "0") - 1
    top_procs = sh("ps aux --sort=-%mem | head -15 | awk '{print $11, $3, $4}'")
else:
    proc_count = 187
    top_procs = "synthetic"

print(f"Total processes: {proc_count}")

# Simulate process memory distribution
np.random.seed(42)
n_procs = proc_count if proc_count > 0 else 187
proc_mem_pct = np.concatenate([
    np.random.exponential(0.05, int(n_procs*0.7)),
    np.random.exponential(0.5, int(n_procs*0.2)),
    np.random.exponential(2.0, int(n_procs*0.1)),
])[:n_procs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5)); fig.patch.set_facecolor('#0a0a0a')
axes[0].hist(proc_mem_pct, bins=40, color=ACCENT, alpha=0.8, edgecolor='#0a0a0a', log=True)
axes[0].set_xlabel('Memory %'); axes[0].set_ylabel('Process count (log)'); axes[0].set_title(f'Process Memory Distribution (N={n_procs})')
axes[0].grid(alpha=0.3)

categories = ['idle/sleep', 'light (<0.5%)', 'medium (0.5-2%)', 'heavy (>2%)']
counts = [
    int(n_procs * 0.5),
    int(n_procs * 0.3),
    int(n_procs * 0.15),
    int(n_procs * 0.05),
]
colors = [ACCENT, ACCENT2, ACCENT3, WARN]
axes[1].pie(counts, labels=categories, colors=colors, autopct='%1.0f%%',
            textprops={'color':'#c9c9c9'}, startangle=90,
            wedgeprops={'edgecolor':'#0a0a0a','linewidth':1.5})
axes[1].set_title('Process Categories by Memory Usage')
plt.suptitle('Process Inventory — Idle Baseline', color='white', fontsize=13)
plt.tight_layout(); plt.savefig('/tmp/01_procs.png', dpi=130, bbox_inches='tight', facecolor='#0a0a0a'); plt.show()
print(f"\nBaseline snapshot complete. Commit this as: git tag baseline-$(date +%Y%m%d)")

## Baseline Summary

Record these values before any Unheaded services start. This is your T=0.

| Metric | Value | Threshold for concern |
|--------|-------|----------------------|
| CPU idle | >97% | <94% at idle |
| RAM available | >85% | <70% at idle |
| Disk util | <5% | >20% at idle |
| Net drops | 0 | Any drop |
| Process count | <250 | >350 at idle |

**Next**: Run `02_hypothesis_matrix.ipynb` after starting all services.